In [ ]:
# Standard libraries

import sys, math, itertools, warnings, importlib, textwrap, random, ast, re, gc, pickle, json, os, sklearn, xgboost, joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
NOTEBOOK_UTILS_SRC = PROJECT_ROOT / "notebook_utils" / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

if str(NOTEBOOK_UTILS_SRC) not in sys.path:
    sys.path.append(str(NOTEBOOK_UTILS_SRC))

from paths import DATA_RAW, DATA_INTERMEDIATE, FIGURES, MODELS, OUTPUTS, TABLES

import numpy as np
import pandas as pd
import operator
from datetime import datetime, date
from IPython.display import display, HTML
import pyfolio as pf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t, norm, entropy
from scipy.optimize import minimize
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split, TimeSeriesSplit, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb

warnings.filterwarnings('ignore')

# 3.7.16 specific
from typing import List

In [ ]:
# Panda display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None) # Show all content of each column
pd.set_option('display.width', 1000)        # Set the display width to 1000 characters
pd.options.display.float_format = '{:,.5f}'.format
np.set_printoptions(precision=5, suppress=True)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)
print("sklearn:", sklearn.__version__)
print("xgb:", xgboost.__version__)

### - Lib import

In [ ]:
# My Functions: same directory
model = 'regime_model'
model_round = 'round_1'

model_input_path = DATA_RAW / "xls" / "input" / model
model_round_input_path = model_input_path / model_round
model_intermediate_path = DATA_INTERMEDIATE / model / model_round
model_table_output_path = TABLES / model / model_round 
model_csv_output_path = model_table_output_path / "csv"
model_xls_output_path = model_table_output_path / "xls"
model_figure_output_path = FIGURES / model / model_round
model_ml_output_path = MODELS / model / model_round
research_liquidity_path = DATA_RAW / "research" / "liquidity" / "xls"
research_fear_greed_path = DATA_RAW / "research" / "fear_greed" / "csv"

import notebook_utils._formatting_functions as _formatting_functions
importlib.reload(_formatting_functions)
from notebook_utils._formatting_functions import (event_import, generate_data_dictionary, enrich_data_dictionary,)


# 0.1 Data Dictionary Creation

In [ ]:
# Imports column names (ie. variables) and creates a dictionary: saved as "pre dictionary"
# using just one file as all the input data files have the same structure

in_file = "BoMoS_v2_1min_a 02-24 20_41_30_010220_022026.csv"

dictionary_df = generate_data_dictionary(input_filename=in_file, input_folder=model_round_input_path, output_folder=model_intermediate_path)

In [ ]:
# Imports master data dictionary, to assign additional categorical tags for processing: saved as "post dictionary"

master_data = "master_variable_inventory 20251223.xlsx"
master_path = model_round_input_path / master_data

assert master_path.exists(), f"Missing master dictionary: {master_path}"

final_df = enrich_data_dictionary(
    current_dictionary_df=dictionary_df,
    master_inventory_path=master_path,
    output_folder=model_intermediate_path,
)

# 1.0 Importing Data From Backtest


In [ ]:
# Data columns by groups: front(most important), kite (always the same), sym_ta_cols (ta colums for symbol), other symbol ta (extra_sym_cols), add-ons (extra), etc..
# 8 Lists of columns are created: front_cols, kite_cols, sym_ta_cols, extra_sym_cols, sec_cols, extra_cols, drop_cols, last_cols
front_cols = None
kite_cols = None
sym_ta_cols = None
extra_sym_cols = None
extra_cols = None
sec_cols = None
last_cols = None
drop_cols = None

grouped = final_df.groupby('code')['kite name'].apply(list)
list_dict = grouped.to_dict()
for key, value in list_dict.items():
    globals()[key] = value

# Important step: Ensures 'symbol' and 'mtm_pl' are first on the list... which will ultimately be 'normed_date' and 'symbol' are the top columns
front_cols.sort(reverse=True)
#['symbol', 'mtm_pl', 'matched_shares', 'entry_pl', 'entry_collect.Daily_open']
print(grouped)

In [53]:
# MAIN RAW TRADE DATA IMPORT
# path = str(model_round_input_path) + "\\"
path = str(model_round_input_path) + os.sep

filenames = [
            'BoMoS_v2_1min_a 02-24 20_41_30_010220_022026.csv',
            'BoMoS_v2_1min_b 02-25 16_13_11_010213_123119.csv',
             ]

event_data = pd.DataFrame()

for file in filenames:
    new_data = event_import(path, file, front_cols, kite_cols, sym_ta_cols, extra_sym_cols, extra_cols, sec_cols, None, drop_cols)
    event_data = pd.concat([event_data, new_data], ignore_index = False)

event_data.sort_values(by=['entry_time', 'symbol'], ascending=[False, True], inplace=True) #Sorts descending by vars

In [ ]:
print(f'observations: {event_data.shape[0]}')

In [ ]:
event_data.tail()

In [ ]:
# 146 initial variables imported / 13624 observations

# Creating a placeholder at the end of the data - dashboard function drops last column of every df - this is a legacy from the previous runs that had this data available (data will be added in future runs)
event_data['modelspec'] = None

columns_list = event_data.columns.tolist()
print(f'variables: {len(columns_list)}')
print(textwrap.fill(", ".join(columns_list), width = 250))
print(f'observations: {event_data.shape[0]}')

### - Data export

In [ ]:
DEBUG_ROWS = 1000  # None = full dataset, used only for notebook refactor/testing
outname = f"event_data {datetime.now().strftime('%Y%m%d')}.csv"
event_data.head(DEBUG_ROWS).to_csv(model_intermediate_path / outname, index=False)

In [ ]:
print(f'observations: {event_data.shape[0]}')

# 2.0 Analysing Raw Data

### - Lib import

In [ ]:
# For dashboards

import notebook_utils._dashboard_functions_one_symbol_v2 as _dashboard_functions_one_symbol_v2
importlib.reload(_dashboard_functions_one_symbol_v2)
from notebook_utils._dashboard_functions_one_symbol_v2 import dashboard

# For deciles, charts and distances
import notebook_utils._data_explore_functions as _data_explore_functions
importlib.reload(_data_explore_functions)
from notebook_utils._data_explore_functions import (
    cross_tabs,
    explore_cross,
    decile_summary,
    line_chart_grid,
    create_distance,
    append_summary,
    get_summary,
    reset_summary,
    count_outliers_by_std,
)

### - Dashboard on full data: Note this is for a SHORT (selling) model

In [ ]:
## Creating Dashboard
# Unoptimized data
LONG_SHORT = -1
ENTRY_FEE =  3.5

# Runding dashboard function
strat_000, pnl_symboldate_000, pnl_bydate_000 = dashboard(event_data, ENTRY_FEE, 'no_optmz', LONG_SHORT, 'complete', 'max')
print(f'full data: {len(event_data)}')
print(f'new data: {len(pnl_symboldate_000)}')

# FOR CODEX: "NOTE: Gross P/L calc matches original" in output is important to reproduce, if not aggregation proocess is not successful


### - Data export

In [ ]:
## Data export for review
outname = f"pnl_symboldate_000 {datetime.now().strftime('%Y%m%d')}.xlsx"
pnl_symboldate_000.head(1000).to_excel(model_intermediate_path / outname, index=False, engine='openpyxl')

outname = f"pnl_date_000 {datetime.now().strftime('%Y%m%d')}.xlsx"
pnl_bydate_000.to_excel(model_intermediate_path / outname, index=False, engine='openpyxl')


### - Data copy 1

In [ ]:
# Copying data as we are adding new variables (11,454 obs in new data pull for ML) - to 2015
# Data copy 1
static_rvw_001 = pnl_symboldate_000.copy()
print(f'observations: {len(static_rvw_001)}')


### 2.1.0 Distance Variables: Symbol related

In [ ]:

# Symbol distance variables (59)
df_symbol_dist = final_df[(final_df['distance'] == True) & (final_df['category'] == 'symbol')][['clean name']]
symbol_dist_cols = df_symbol_dist['clean name'].tolist()
symbol_dist_cols += ["qtd", "mtd"]

print(f'symbol: {len(symbol_dist_cols)}')
print(textwrap.fill(", ".join(symbol_dist_cols), width = 250))

# Spy distance variables (23)
df_spy_dist = final_df[(final_df['distance'] == True) & (final_df['category'] == 'spy')][['clean name']]
spy_dist_cols = df_spy_dist['clean name'].tolist()
print(f'spy: {len(spy_dist_cols)}')
print(textwrap.fill(", ".join(spy_dist_cols), width = 250))


In [ ]:
# Uses calculated ATR (TA.Lib) to normalize, and the 'ask' as reference variable
for column in symbol_dist_cols:
    # to the ask
    create_distance('dist', static_rvw_001, 'prev_close', column, 'atr_ktg');
    # gap 1: open to prev close
    create_distance('dist', static_rvw_001, 'day_open', 'prev_close', 'atr_ktg');
    # gap 2: open to premarket high
    create_distance('dist', static_rvw_001, 'day_open', 'premkt_m_h', 'atr_ktg');

# 59 vars
symbol_dist_cols = [col for col in static_rvw_001.columns if col.startswith('dist_')]

print(len(symbol_dist_cols))
print(textwrap.fill(", ".join(symbol_dist_cols), width = 250))


In [ ]:
# Need to get clean list of distance variables from original data - ie. make sure dist_xxxx variables are NOT in this list as I am calculating other form of distance
df_symbol_dist = final_df[(final_df['distance'] == True) & (final_df['category'] == 'symbol')][['clean name']]
symbol_dist_cols = df_symbol_dist['clean name'].tolist()
symbol_dist_cols += ["qtd", "mtd"]

# Relative distances over reference px (not ATR)
for column in symbol_dist_cols:
    # to the ask
    create_distance('pct', static_rvw_001, 'prev_close', column, 'prev_close');
    # gap 1: open to prev close
    create_distance('pct', static_rvw_001, 'day_open', 'prev_close', 'day_open');
    # gap 2: open to premarket high
    create_distance('pct', static_rvw_001, 'day_open', 'premkt_m_h', 'day_open');

# 59 vars
symbol_pct_cols = [col for col in static_rvw_001.columns if col.startswith('pct_')]

print(len(symbol_pct_cols))
print(textwrap.fill(", ".join(symbol_pct_cols), width = 250))


### 2.2.0 Distance Variables: SPY related

In [ ]:
# 22 SPY distance variables with the addition of daily_open (as of 1/7/2025)
# Removing reference variable for the distance
spy_dist_cols.remove('spy_trade_px')

for column in spy_dist_cols:
    # to the SPY level at entry
    create_distance('dist', static_rvw_001, 'spy_trade_px', column, 'spy_atr');
    
spy_dist_cols = [col for col in static_rvw_001.columns if col.startswith('dist_spy')]
print(len(spy_dist_cols))
print(textwrap.fill(", ".join(spy_dist_cols), width = 250))


In [ ]:
# Spy distance variables (22): same process, repopulating clean list of SPY distances
df_spy_dist = final_df[(final_df['distance'] == True) & (final_df['category'] == 'spy')][['clean name']]
spy_dist_cols = df_spy_dist['clean name'].tolist()
spy_dist_cols.remove('spy_trade_px')

for column in spy_dist_cols:
    # to the SPY level at entry
    create_distance('pct', static_rvw_001, 'spy_trade_px', column, 'spy_atr');
    
spy_pct_cols = [col for col in static_rvw_001.columns if col.startswith('pct_spy')]
print(len(spy_pct_cols))
print(textwrap.fill(", ".join(spy_pct_cols), width = 250))


### 2.4.0 Consolidating Distances: Symbol + SPY

### - Distance columns

In [ ]:
symbol_dist_cols = [col for col in static_rvw_001.columns if col.startswith('dist_prev')]
symbol_dist_cols += ["dist_day_open_prev_close", "dist_day_open_premkt_m_h"]
print(len(symbol_dist_cols))
print(textwrap.fill(", ".join(symbol_dist_cols), width = 250))

spy_dist_cols = [col for col in static_rvw_001.columns if col.startswith('dist_spy')]
print(len(spy_dist_cols))
print(textwrap.fill(", ".join(spy_dist_cols), width = 250))


In [ ]:
# 83 variables as of 12/24 (added relative distances)

consolidated_dist = symbol_dist_cols + spy_dist_cols
print(len(consolidated_dist))
print(textwrap.fill(", ".join(consolidated_dist), width = 250))

### - Percent columns

In [ ]:
consolidated_pct = symbol_pct_cols + spy_pct_cols
print(len(consolidated_pct))
print(textwrap.fill(", ".join(consolidated_pct), width = 250))

In [ ]:
# Initialize an empty DataFrame to store the consolidated results
# relative distance ratios are very small (dist_ask_spy_prev_close, dist_ask_spy_open)
dist_consol_summ = pd.DataFrame()

# Loop over each variable and compute the decile summary
for var in consolidated_dist:
    summary = decile_summary(static_rvw_001, var)
    dist_consol_summ[var] = summary  # Store the summary in the DataFrame with the variable name as the column

dist_consol_summ


In [ ]:
# Initialize an empty DataFrame to store the consolidated results
# relative distance ratios are very small (dist_ask_spy_prev_close, dist_ask_spy_open)
pct_consol_summ = pd.DataFrame()

# Loop over each variable and compute the decile summary
for var in consolidated_pct:
    summary = decile_summary(static_rvw_001, var)
    pct_consol_summ[var] = summary  # Store the summary in the DataFrame with the variable name as the column

pct_consol_summ


### 2.5.0 Volume Variable distribution

In [ ]:
## Volume variable ratios: useful vars collected in list
# NOTE: s1_vol_test is just 1% of d_avol250

# Accumulated Less Core Volume - THESE VARIABLE CAN BE DELETED!!!
static_rvw_001['open_vol'] = (static_rvw_001['vol_acc'] - static_rvw_001['vol_core'])

# Collecting relevant volume variables (5): 'd_avol250' left out as scaling factor
volu_vars = ['d_avol5', 'd_avol50', 'premkt_vol', 'vol_acc', 'open_vol']

# Creating volume ratios (Using 250-day Average Vol)
for item in volu_vars:
    static_rvw_001[f'{item}_rat'] = static_rvw_001[item] / static_rvw_001['d_avol250']

list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')
print(textwrap.fill(", ".join(list_vars), width = 250))

# variables:328
# observations:11376


### - Adding RVOL ratios

In [ ]:
# Adding relative RVOLS
static_rvw_001['symb_spy_rvol_rat'] = static_rvw_001['rvol'] / static_rvw_001['spy_rvol']

# Analysis of all volume variables (13)
volu_vars = ['rvol', 'd_avol5', 'd_avol50', 'premkt_vol', 'vol_acc', 'd_avol5_rat', 'd_avol50_rat', 'premkt_vol_rat', 'vol_acc_rat', 'symb_spy_rvol_rat', 'open_vol_rat']

print(len(volu_vars))
print(volu_vars)

volu_consol_dist = pd.DataFrame()

# Loop over each variable and compute the decile summary
for var in volu_vars:
    summary = decile_summary(static_rvw_001, var)
    volu_consol_dist[var] = summary  # Store the summary in the DataFrame with the variable name as the column

volu_consol_dist


In [ ]:
outlier_table = count_outliers_by_std(static_rvw_001, volu_vars)
print(outlier_table)

### 2.8.0 Time Variables

In [ ]:
# 339 variables to this point (with the addition of ATR category and time variables)

static_rvw_001['entry_hr_dec'] = static_rvw_001['entry_time'].dt.hour + static_rvw_001['entry_time'].dt.minute / 60
static_rvw_001['exit_hr_dec'] = static_rvw_001['exit_time'].dt.hour + static_rvw_001['exit_time'].dt.minute / 60
static_rvw_001['entry_hr_dec_to_close'] = 16.00 - static_rvw_001['entry_hr_dec']

static_rvw_001["year_day"]  = static_rvw_001["entry_time"].dt.dayofyear

static_rvw_001["week_day_sin"] = np.sin(2 * np.pi * static_rvw_001["week_day"] / 7)
static_rvw_001["week_day_cos"] = np.cos(2 * np.pi * static_rvw_001["week_day"] / 7)

static_rvw_001["month_sin"] = np.sin(2 * np.pi * static_rvw_001["month"] / 12)
static_rvw_001["month_cos"] = np.cos(2 * np.pi * static_rvw_001["month"] / 12)

static_rvw_001["year_day_sin"] = np.sin(2 * np.pi * static_rvw_001["year_day"] / 365.25)
static_rvw_001["year_day_cos"] = np.cos(2 * np.pi * static_rvw_001["year_day"] / 365.25)

list_vars = static_rvw_001.columns.tolist()

print(len(list_vars))
print(len(static_rvw_001))
print(textwrap.fill(", ".join(list_vars), width = 250))

### 2.9.0 Other Variables: Normalized day range, 2-day VWAP and Change from open

In [ ]:
# 341 variables
static_rvw_001['dist_day_range'] = static_rvw_001['day_range'] / static_rvw_001['atr_ktg']

# Change from Open
# static_rvw_001['dist_change_from_open'] = (static_rvw_001['change_from_open']) / static_rvw_001['atr']

# Relative spread
static_rvw_001['spread_perc'] = (static_rvw_001['spread_nbbo']) / static_rvw_001['entry_price']


list_vars = static_rvw_001.columns.tolist()

print(len(list_vars))
print(len(static_rvw_001))
print(textwrap.fill(", ".join(list_vars), width = 250))

static_rvw_001.head(2)

### 2.10.0 Return variables

#### - Calendar variables

In [ ]:
prev_close_vars = [
            'mtd', 'qtd', 'ytd', 
            'px_1m_ago', 'px_2m_ago', 'px_3m_ago', 'px_4m_ago', 'px_5m_ago', 'px_6m_ago', 'px_7m_ago', 'px_8m_ago', 'px_9m_ago', 'px_10m_ago', 'px_11m_ago', 'px_12m_ago', 
            'px_prev2', 'px_prev3', 'px_prev4', 'px_prev5', 'px_prev6', 'px_prev10', 
            'prev_high', 'prev_low', 'prev_vwap'
        ]

# creates returns from prev_close to the last 9 day closing
for item in prev_close_vars:
    static_rvw_001[f'ret_{item}'] = (static_rvw_001[item] / static_rvw_001['prev_close'])-1

# 366 variables to this point / 11376 obs
list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')



#### - Moving averages

In [ ]:
ma_vars = ['d_ema8', 'd_ema20', 'd_ema50', 'd_ema100', 'd_ema200', 'ema','sma', 'kama',  'kalmar_f']

# creates returns from prev_close to the last 9 day closing
for item in ma_vars:
    static_rvw_001[f'ret_{item}'] = (static_rvw_001[item] / static_rvw_001['prev_close'])-1

# 375 variables to this point / 11376 obs
list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')

#### - TS Forecasts

In [ ]:
# ts_vars = ['arimax1', 'arimax2', 'var1', 'var2', 'bb_low', 'bb_up']
ts_vars = ['arimax1', 'arimax2', 'var1', 'var2']

for item in ts_vars:
    static_rvw_001[f'dumm_{item}'] = (static_rvw_001[item] > static_rvw_001['prev_close']).astype(int)

# 381 variables to this point / 11376 obs
list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')

### 2.11.0 External variables

#### - Liquidity indices

In [ ]:
# Importing master list of columns/features

vars_name = f'liquidity_indices_updated_20260309.xlsx'
cols = ['DATE', 'PCA_Index_ma5', 'PCA_ScaledIndex_ma5', 'PCA_Index_ma20', 'PCA_ScaledIndex_ma20', 'PCA_Index_ma50', 'PCA_ScaledIndex_ma50', 'PCA_Raw_full', 'PCA_Index_full']

liquidity_df = pd.read_excel(research_liquidity_path / vars_name, sheet_name='Sheet1', usecols=cols, parse_dates=["DATE"])

liquidity_df = liquidity_df.dropna(how="any")
liquidity_df["normed_date"] = liquidity_df["DATE"].dt.strftime("%Y-%m-%d")  # or any format


In [ ]:
# Must normalize the column types, as object masks the true type of the column, thus merge WILL FAIL
static_rvw_001["normed_date"] = pd.to_datetime(static_rvw_001["normed_date"]).dt.normalize()
liquidity_df["normed_date"] = pd.to_datetime(liquidity_df["normed_date"]).dt.normalize()

static_rvw_002 = static_rvw_001.merge(
    liquidity_df.drop(columns=["DATE"]),
    how="left",
    on="normed_date",
    suffixes=("", "_liq")
)

print(static_rvw_002[["normed_date", "PCA_Index_full"]].head(1))
print(static_rvw_002[["normed_date", "PCA_Index_full"]].tail(1))

#### - Fear and Greed

In [ ]:
# Importing master list of columns/features
vars_name = f'fear_and_greed_full.csv'
cols = ['date', 'value']

fear_df = pd.read_csv(research_fear_greed_path / vars_name, usecols=cols)
fear_df = fear_df.dropna(how="any")
fear_df = fear_df.rename(columns={"date": "normed_date", "value": "fear_greed"})


In [ ]:
fear_df["normed_date"] = pd.to_datetime(fear_df["normed_date"]).dt.normalize()

static_rvw_003 = static_rvw_002.merge(
    fear_df,
    how="left",
    on="normed_date",
    suffixes=("", "_liq")
)

print(static_rvw_003[["normed_date", "fear_greed"]].head(1))
print(static_rvw_003[["normed_date", "fear_greed"]].tail(1))

In [ ]:
list_vars = static_rvw_003.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_003)}')
print(textwrap.fill(", ".join(list_vars), width = 250))

In [ ]:
summary = (
    static_rvw_003.isna().sum()
    .to_frame("missing")
    .assign(
        dtype=static_rvw_003.dtypes,
        non_null=lambda x: len(static_rvw_003) - x["missing"],
        percent_missing=lambda x: x["missing"] / len(static_rvw_003) * 100,
        unique=static_rvw_003.nunique()))


In [ ]:
missing_list = summary[summary["missing"] > 0].index.tolist()
missing_list

In [ ]:
outname = f"data_review_{datetime.now().strftime('%Y%m%d')}.csv"
summary[summary["missing"] == 0].to_csv(model_intermediate_path / outname, index=True)

In [ ]:
gc.collect()

# 3.0 Data Cleaning

### 3.0.1 Data copy 3

In [ ]:
# Base Data for Static Optimization
# Deleting variables with missing rows (in this case 12-mo prices are missing 48 rows)

static_rvw_003.drop(columns=missing_list, inplace=True)

print(f"Dropping {len(missing_list)} columns with missing values")
print(textwrap.fill(", ".join(missing_list), width=250))

partition_a = static_rvw_003.copy()

In [ ]:
reset_summary()

summary_df = append_summary(static_rvw_001, "static_rvw_001")
summary_df = append_summary(static_rvw_002, "static_rvw_002")
summary_df = append_summary(static_rvw_003, "static_rvw_003")
summary_df = append_summary(partition_a, "partition_a")
print(summary_df)

### 3.0.4 Creating 80/20 split of partition a (full data): INS and OOS

In [ ]:
break_pct = 0.80
sample = int(round(len(partition_a) * break_pct, 0))
sample_breakdate = partition_a.iloc[sample]['normed_date']
print(sample_breakdate)

partition_ins = partition_a[partition_a['normed_date'] < sample_breakdate]
partition_oos = partition_a[partition_a['normed_date'] >= sample_breakdate]

print(len(partition_ins))
print(partition_ins['normed_date'].tail(2))
print("")
print(len(partition_oos))
print(partition_oos['normed_date'].head(2))


### 3.0.5 Creating 80/20 split of INS

In [ ]:
sample = int(round(len(partition_ins) * break_pct, 0))
sample_breakdate = partition_ins.iloc[sample]['normed_date']
print(sample_breakdate)

partition_ins_80 = partition_ins[partition_ins['normed_date'] < sample_breakdate]
partition_ins_20 = partition_ins[partition_ins['normed_date'] >= sample_breakdate]

print(len(partition_ins_80))
print(partition_ins_80['normed_date'].head(1))
print(partition_ins_80['normed_date'].tail(1))
print("")
print(len(partition_ins_20))
print(partition_ins_20['normed_date'].head(1))
print(partition_ins_20['normed_date'].tail(1))


In [ ]:
# INS/OOS
summary_df = append_summary(partition_ins, "partition_ins")
summary_df = append_summary(partition_oos, "partition_oos")
# INS (train/test)
summary_df = append_summary(partition_ins_20, "partition_ins_20")
summary_df = append_summary(partition_ins_80, "partition_ins_80")
print(summary_df)

#### - Saving pre-optimized data (with all features)

In [ ]:
DEBUG_ROWS = 50000   # None = full dataset

dfs = {
    "partition_ins_80": partition_ins_80,
    "partition_ins_20": partition_ins_20,
    "partition_oos":    partition_oos,
    "partition_ins":    partition_ins,
}

for name, df in dfs.items():

    df_out = df if DEBUG_ROWS is None else df.head(DEBUG_ROWS)

    df_out.to_parquet(
        model_intermediate_path / f"{name}.parquet",
        engine="pyarrow",
        index=False,
        compression="zstd"
    )

    print(f"{name}: saving {len(df_out)} rows (original {len(df)})")